## Face Recognition using Webcam Video
- Captures video from webcam using `OpenCV`
- Detects and recognizes faces using the `face_recognition`
- Shows recognized faces in real-time
- Saves screenshots of recognized faces


In [1]:
import face_recognition
import cv2
import numpy as np
import os
from datetime import datetime

In [2]:
def load_known_faces(known_people_folder):
    known_face_encodings = []
    known_face_names = []
    
    # Traverse the directory containing known people
    for person_name in os.listdir(known_people_folder):
        person_folder = os.path.join(known_people_folder, person_name)
        
        if os.path.isdir(person_folder):
            print(f"Loading images for: {person_name}")
            
            # Read each image file in person's folder
            for image_file in os.listdir(person_folder):
                image_path = os.path.join(person_folder, image_file)
                
                # Skip if not a valid image file
                if not os.path.isfile(image_path):
                    continue
                    
                try:
                    # Load image and create encoding
                    person_image = face_recognition.load_image_file(image_path)
                    encodings = face_recognition.face_encodings(person_image)
                    
                    if encodings:
                        known_face_encodings.append(encodings[0])
                        known_face_names.append(person_name)
                        print(f"Loaded {image_file}")
                except Exception as e:
                    print(f" Error loading {image_file}: {e}")
    
    print(f"\nTotal known faces loaded: {len(known_face_encodings)}")
    print(f"Total people: {len(set(known_face_names))}")
    return known_face_encodings, known_face_names

# Load the known faces (including Elon Musk, Mark Zuckerberg and me)
known_face_encodings, known_face_names = load_known_faces("known_people")

Loading images for: Mark Zuckerberg
Loaded zuckerberg3.png
Loaded zuckerberg2.png
Loaded zuckerberg1.png
Loading images for: Elon Musk
Loaded side.png
Loaded ElonMusk.PNG
Loaded elon musk side.png
Loading images for: Nghia Tran
Loaded 9F0D1986-B31B-494D-987A-F71BE538BAA9.jpeg
Loaded 438399A0-F902-4021-A5BA-2F8C2383E21A_1_105_c.jpeg
Loaded 105E619F-B39A-4E35-8D64-2227EC8043C3_1_105_c.jpeg

Total known faces loaded: 9
Total people: 3


In [ ]:
# Real-time face detection and recognition
# Draws bounding boxes around detected faces
# Shows person names for recognized faces

def webcam_face_recognition(known_face_encodings, known_face_names, tolerance=0.5, save_images=True):

    # Open webcam
    video_capture = cv2.VideoCapture(0)
    
    if not video_capture.isOpened():
        print("Could not open webcam")
        return
    
    print("Webcam opened.")
    
    # Counter for saving images
    saved_images_count = 0
    max_saved_images = 1
    
    while True:
        # Get a single frame of video
        ret, frame = video_capture.read()
        
        if not ret:
            print("Failed to grab frame cmnr")
            break
        
        # Resize frame of video to 1/4 size for faster face recognition processing (recommended)
        small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
        
        # Convert BGR (OpenCV) to RGB (face_recognition uses RGB)
        rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
        
        # Find all the faces and face encodings in the current frame of video
        face_locations = face_recognition.face_locations(rgb_small_frame)
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)
        
        face_names = []
        for face_encoding in face_encodings:
            # See if the face is a match for the known face(s)
            matches = face_recognition.compare_faces(
                known_face_encodings, face_encoding, tolerance=tolerance)
            name = "Unknown"
            
            # Use the known face with the smallest distance to the new face
            face_distances = face_recognition.face_distance(
                known_face_encodings, face_encoding)
            
            if len(face_distances) > 0:
                best_match_index = np.argmin(face_distances)
                if matches[best_match_index]:
                    name = known_face_names[best_match_index]
                    confidence = 1 - face_distances[best_match_index]
                    
                    # Save image if this is a known person and i haven't saved 3 yet
                    if save_images and saved_images_count < max_saved_images:
                        # Find the face location for this specific face
                        face_idx = face_encodings.index(face_encoding)
                        (top, right, bottom, left) = face_locations[face_idx]
                        
                        # Scale back up face locations since the frame i detected in was scaled to 1/4 size
                        top *= 4
                        right *= 4
                        bottom *= 4
                        left *= 4
                        
                        # Draw box and save
                        face_image = frame[top:bottom, left:right]
                        if face_image.size > 0:
                            filename = f"recognized_face_{name}.jpg"
                            cv2.imwrite(filename, face_image)
                            saved_images_count += 1
                            print(f"Saved imag to {filename}")
            
            face_names.append(name)
        
        # Display the results
        for (top, right, bottom, left), name in zip(face_locations, face_names):
            # Scale back up face locations since the frame i detected in was scaled to 1/4 size
            top *= 4
            right *= 4
            bottom *= 4
            left *= 4
            
            # Draw a box around the face
            if name != "Unknown":
                cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
                cv2.rectangle(frame, (left, bottom - 35), (right, bottom), (0, 255, 0), cv2.FILLED)
                font = cv2.FONT_HERSHEY_DUPLEX
                cv2.putText(frame, name, (left + 6, bottom - 6), font, 0.6, (255, 255, 255), 1)
            else:
                cv2.rectangle(frame, (left, top), (right, bottom), (0, 0, 255), 2)
                cv2.rectangle(frame, (left, bottom - 35), (right, bottom), (0, 0, 255), cv2.FILLED)
                font = cv2.FONT_HERSHEY_DUPLEX
                cv2.putText(frame, name, (left + 6, bottom - 6), font, 0.6, (255, 255, 255), 1)
        
        # Display the resulting image
        cv2.imshow('Face Recognition', frame)
        
        # Click on "q" to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    # Release handle to the webcam
    video_capture.release()
    cv2.destroyAllWindows()
    print("Camera released.")
    print(f"Total images saved: {saved_images_count}")

# tolerance: How much distance to allow when matching faces (lower = stricter)
webcam_face_recognition(known_face_encodings, known_face_names, tolerance=0.5, save_images=True)

Webcam opened.
Saved imag to recognized_face_Nghia Tran.jpg
Camera released.
Total images saved: 1


: 